# Example: TensorBoard for Chest X-Ray Classifier Training

This notebook complements **[Week6_Pneumonia_Assignment.ipynb](Week6_Pneumonia_Assignment.ipynb)**. It uses the same Lightning data module, ResNet-18 head, and optimizer, but wires in **TensorBoard** so you can watch optimization while you train. Training runs for a **fixed number of epochs** (no early stopping) so the dashboard shows full curves; your assignment still stops when validation accuracy reaches 80%.

## What you will learn

1. Attach Lightning's `TensorBoardLogger` to a `Trainer`
2. Log training and validation metrics from a `LightningModule`
3. Log metrics at **epoch index** so TensorBoard x-axis reads 0, 1, 2, …
4. Open TensorBoard inside the notebook (or from the terminal)
5. Compare two hyperparameter settings in the same dashboard

> Complete the assignment notebook first if you want practice implementing the modules yourself. Here, the model code is filled in so you can focus on logging and visualization.

## Setup

Same downloads as the assignment, plus `tensorboard` for the dashboard.

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "lightning",
        "torchmetrics",
        "tensorboard",
    ],
    check=True,
)

COURSE_RAW = (
    "https://raw.githubusercontent.com/opencampus-sh/course-material/main"
    "/applied-machine-learning/week-06"
)
HF_DATASET = (
    "https://huggingface.co/datasets/opencampus/chest-xray-pneumonia-3class-balanced"
    "/resolve/main"
)

VIZ_FILE = Path("xray_viz.py")
WEIGHTS_FILE = Path("resnet18_chest_xray_classifier_weights.pth")
DATA_MARKER = Path("chest_xray/train/NORMAL")


def wget(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-q", "-c", "-O", str(destination), url], check=True)


if not VIZ_FILE.exists():
    print("Downloading xray_viz.py from GitHub...")
    wget(f"{COURSE_RAW}/xray_viz.py", VIZ_FILE)

if not DATA_MARKER.exists():
    zip_path = Path("chest_xray_prepared.zip")
    print("Downloading prepared dataset from Hugging Face (~1 GB)...")
    wget(f"{HF_DATASET}/chest_xray_prepared.zip", zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(".")
    zip_path.unlink(missing_ok=True)

if not WEIGHTS_FILE.exists():
    print("Downloading pretrained weights from Hugging Face...")
    wget(f"{HF_DATASET}/resnet18_chest_xray_classifier_weights.pth", WEIGHTS_FILE)

print("Setup complete.")

In [ ]:
import os
from pathlib import Path

import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.optim as optim
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torchmetrics.classification import Accuracy
from torchvision import datasets, models as tv_models, transforms

import xray_viz

torch.set_float32_matmul_precision("medium")

DATA_DIR = "./chest_xray/"
BACKBONE_WEIGHTS = "./resnet18_chest_xray_classifier_weights.pth"
LOG_ROOT = Path("tb_logs")

## Model and data (same as the assignment solution)

The blocks below match the completed Week 6 assignment. The only change for TensorBoard is extra `self.log(...)` calls in the Lightning module.

In [ ]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(xray_viz.NORMALIZE_MEAN, xray_viz.NORMALIZE_STD),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(xray_viz.NORMALIZE_MEAN, xray_viz.NORMALIZE_STD),
])


def make_imagefolder_datasets(train_path, val_path, train_tf, val_tf):
    train_ds = datasets.ImageFolder(train_path, train_tf)
    val_ds = datasets.ImageFolder(val_path, val_tf)
    return train_ds, val_ds


def make_dataloader(train_ds, val_ds, batch_size, for_training):
    dataset = train_ds if for_training else val_ds
    return DataLoader(dataset, batch_size=batch_size, shuffle=for_training)


class XRayDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=64):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.train_transform = TRAIN_TRANSFORM
        self.val_transform = VAL_TRANSFORM
        self.train_dataset = None
        self.val_dataset = None

    def setup(self, stage=None):
        train_path = os.path.join(self.data_dir, "train")
        val_path = os.path.join(self.data_dir, "val")
        self.train_dataset, self.val_dataset = make_imagefolder_datasets(
            train_path, val_path, self.train_transform, self.val_transform
        )

    def train_dataloader(self):
        return make_dataloader(
            self.train_dataset, self.val_dataset, self.batch_size, for_training=True
        )

    def val_dataloader(self):
        return make_dataloader(
            self.train_dataset, self.val_dataset, self.batch_size, for_training=False
        )


def build_resnet_backbone(num_classes, weights_path):
    backbone = tv_models.resnet18(weights=None)
    in_features = backbone.fc.in_features
    backbone.fc = nn.Linear(in_features, num_classes)
    state_dict = torch.load(weights_path, map_location="cpu")
    backbone.load_state_dict(state_dict)
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for parameter in backbone.fc.parameters():
        parameter.requires_grad = True
    return backbone


def build_optimizer_and_scheduler(model, learning_rate, weight_decay):
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=2
    )
    return optimizer, scheduler

### Lightning module with TensorBoard-friendly logging

Lightning forwards anything you pass to `self.log(...)` to the progress bar. We set `logger=False` on those calls and write to TensorBoard from a callback instead (see below) so the x-axis uses **epoch index**.

| Metric | When | TensorBoard tab |
|--------|------|-----------------|
| `train_loss` | end of each training epoch | Scalars |
| `val_loss`, `val_acc` | each validation epoch | Scalars |
| `lr-AdamW` | each epoch | Scalars |
| hyperparameters | after training | HPARAMS (when comparing runs) |

In [ ]:
class PneumoniaLitModule(pl.LightningModule):
    def __init__(
        self,
        weights_path,
        num_classes=3,
        learning_rate=1e-3,
        weight_decay=1e-2,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = build_resnet_backbone(num_classes, weights_path)
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx=None):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_step=False,
            on_epoch=True,
            logger=False,
        )
        return loss

    def validation_step(self, batch, batch_idx=None):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = self.accuracy(logits, y)
        self.log_dict(
            {"val_loss": loss, "val_acc": acc},
            prog_bar=True,
            on_step=False,
            on_epoch=True,
            logger=False,
        )

    def configure_optimizers(self):
        optimizer, scheduler = build_optimizer_and_scheduler(
            self.model, self.hparams.learning_rate, self.hparams.weight_decay
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

## Training helper with TensorBoard

Compared to the assignment's `fit_classifier`, we:

1. Pass a `TensorBoardLogger` instead of `logger=False`
2. Add `EpochScalarLogger` so metrics plot against **epoch index** (0, 1, 2, …)
3. Train for a fixed number of epochs (no early stopping) so curves are visible
4. Call `logger.log_hyperparams(..., metrics=...)` after training for the **HPARAMS** tab

In [ ]:
class EpochScalarLogger(Callback):
    """Write epoch metrics to TensorBoard at epoch index (0, 1, 2, …)."""

    METRICS = ("train_loss", "val_loss", "val_acc")

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking or trainer.logger is None:
            return
        writer = trainer.logger.experiment
        epoch = trainer.current_epoch
        for name in self.METRICS:
            if name not in trainer.callback_metrics:
                continue
            value = trainer.callback_metrics[name]
            scalar = value.detach().float().item() if torch.is_tensor(value) else float(value)
            writer.add_scalar(name, scalar, global_step=epoch)
        if trainer.optimizers:
            lr = trainer.optimizers[0].param_groups[0]["lr"]
            writer.add_scalar("lr-AdamW", lr, global_step=epoch)


def create_early_stopping(num_epochs, accuracy_target):
    return EarlyStopping(
        monitor="val_acc",
        stopping_threshold=accuracy_target,
        patience=num_epochs // 2,
        mode="max",
    )


def fit_with_tensorboard(
    *,
    learning_rate,
    batch_size,
    num_epochs,
    run_name,
    dry_run=False,
    use_early_stopping=False,
    accuracy_target=0.80,
):
    """Train one run and write logs under tb_logs/pneumonia_classifier/<run_name>."""
    logger = TensorBoardLogger(
        save_dir=str(LOG_ROOT),
        name="pneumonia_classifier",
        version=run_name,
        default_hp_metric=False,
    )

    callbacks = [EpochScalarLogger()]
    if use_early_stopping:
        callbacks.append(create_early_stopping(num_epochs, accuracy_target))

    datamodule = XRayDataModule(DATA_DIR, batch_size=batch_size)
    model = PneumoniaLitModule(
        weights_path=BACKBONE_WEIGHTS,
        learning_rate=learning_rate,
    )

    trainer = pl.Trainer(
        max_epochs=num_epochs,
        accelerator="auto",
        devices=1,
        precision="16-mixed",
        logger=logger,
        callbacks=callbacks,
        enable_progress_bar=True,
        enable_model_summary=False,
        enable_checkpointing=False,
        fast_dev_run=dry_run,
    )

    trainer.fit(model, datamodule)

    final_metrics = {}
    for key in ("val_acc", "val_loss", "train_loss"):
        if key not in trainer.callback_metrics:
            continue
        value = trainer.callback_metrics[key]
        final_metrics[key] = (
            value.detach().float().item() if torch.is_tensor(value) else float(value)
        )

    logger.log_hyperparams(
        {
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
        },
        metrics=final_metrics,
    )

    return trainer, model, logger

## Run 1 — baseline learning rate

These settings mirror the assignment's training cell (`batch_size=8`, `lr=1e-3`), but we train for **5 fixed epochs** instead of early stopping at 80% val accuracy so TensorBoard shows curves.

Run the cleanup cell below first so TensorBoard only shows the two runs from this notebook (`lr_1e-3` and `lr_1e-4`).

In [ ]:
import shutil

shutil.rmtree(LOG_ROOT, ignore_errors=True)
print(f"Cleared {LOG_ROOT}/ — fresh runs will be written as lr_1e-3 and lr_1e-4.")

In [ ]:
pl.seed_everything(15)

BATCH_SIZE = 8
NUM_EPOCHS = 5

trainer_lr1e3, model_lr1e3, logger_lr1e3 = fit_with_tensorboard(
    learning_rate=1e-3,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    run_name="lr_1e-3",
)

print(f"Logs written to: {logger_lr1e3.log_dir}")

## Open TensorBoard in the notebook

Run the cell below **after** at least one training run. In Jupyter / Colab / VS Code, the dashboard embeds below the cell.

**Outside the notebook**, from this folder:

```bash
tensorboard --logdir tb_logs
```

Then open the URL printed in the terminal (usually http://localhost:6006).

### What to inspect

Open the **Scalars** tab (not Time Series). Select these tags:

- **train_loss / val_loss**: loss should trend down; a rising `val_loss` while `train_loss` falls can mean overfitting.
- **val_acc**: should climb over epochs 0–4.
- **lr-AdamW**: steps down when `ReduceLROnPlateau` fires on plateauing `val_loss`.

In the runs list on the left, select **`lr_1e-3`** and **`lr_1e-4`**. Numbered folders such as `pneumonia_classifier/3` are leftover runs from earlier sessions — skip them if you ran the cleanup cell.

### Reading the x-axis

Each point is one **epoch** (0, 1, 2, …). After both runs you should see **five points per curve** — not a single dot at step ~500. If you only see one point, you are likely viewing an old run from before the cleanup cell or before this notebook was updated; clear `tb_logs/` and re-run both training cells.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

## Run 2 — compare a lower learning rate

TensorBoard overlays multiple **runs** (subfolders under `tb_logs/pneumonia_classifier/`). Train again with a smaller learning rate, then re-run the TensorBoard cell above so the dashboard picks up both curves.

Compare **`lr_1e-3`** vs **`lr_1e-4`** on the **Scalars** tab.

In [ ]:
pl.seed_everything(15)

trainer_lr1e4, model_lr1e4, logger_lr1e4 = fit_with_tensorboard(
    learning_rate=1e-4,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    run_name="lr_1e-4",
)

print(f"Logs written to: {logger_lr1e4.log_dir}")

## HPARAMS tab (optional)

After both runs finish, open the **HPARAMS** tab in TensorBoard. Select `learning_rate` as the hyperparameter and `val_acc` (or `val_loss`) as the metric to see which setting worked better on this small head-only fine-tuning task.

Final metrics are logged after `trainer.fit` completes — as in `fit_with_tensorboard` above.

## Bring TensorBoard into your assignment

Once your assignment modules work, change **Exercise 4** roughly as follows:

1. `from lightning.pytorch.loggers import TensorBoardLogger`
2. Copy the `EpochScalarLogger` callback from this notebook (or add `LearningRateMonitor` and accept global-step x-axis)
3. Replace `logger=False` with `logger=TensorBoardLogger("tb_logs", name="my_run")`
4. Add `self.log("train_loss", loss, on_step=False, on_epoch=True, logger=False)` in `training_step` if using `EpochScalarLogger`
5. Add `logger=False` to your existing `self.log_dict(...)` in `validation_step` if using `EpochScalarLogger`

When you open TensorBoard, use the **Scalars** tab. With `EpochScalarLogger`, the x-axis is epoch index; with default Lightning logging it is global batch step.

---

**Dataset citation:** Kermany, D.; Zhang, K.; Goldbaum, M. *Cell* 2018. DOI [10.17632/rscbjbr9sj.3](https://doi.org/10.17632/rscbjbr9sj.3)